# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. It follows best practices for referencing dataset entities by their `@id` and provides step-by-step guidance for working with Croissant-formatted datasets in Python.

### Dataset Source
FAIR^2 Croissant-encoded schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Install and import the mlcroissant library
!pip install --quiet mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant` and inspect the high-level description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # access as an object, not as a dict

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nVersion: {getattr(metadata, 'version', None)}")

## 2. Data Overview
Inspect available record sets, their field and column structure, and review their `@id`s. This ensures that subsequent data access references entities according to the Croissant specification.

In [ ]:
# List all record sets by their '@id', name, and available fields.
from mlcroissant._dataset.metadata import RecordSet

record_set_ids = []
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs in metadata.record_sets:
        record_set_ids.append(rs['@id'])
    print("Available Record Sets (by `@id`):")
    for rs in metadata.record_sets:
        print(f"- @id: {rs['@id']}")
        print(f"  Name: {rs.get('name','')}")
        fields = rs.get('fields', [])
        columns = rs.get('columns', [])
        # Print fields and columns in this record set
        print(f"  Fields: {[f['@id'] for f in fields] if fields else 'None'}")
        print(f"  Columns: {[c['@id'] for c in columns] if columns else 'None'}")
        print()
else:
    # If not present, fallback: enumerate using the dataset interface
    print("No record_sets found in explicit metadata. Using dataset.records() interface to discover record sets.")
    discovered = []
    for rs in dataset.record_sets():
        discovered.append(rs['@id'])
        print(f"RecordSet: {rs['@id']}  Name: {rs.get('name','')}")
        flds = rs.get('fields', [])
        print(f"  Fields: {[f['@id'] for f in flds] if flds else 'None'}")
    record_set_ids = discovered

# If record_sets are not defined above, try inferring them from the file objects
if not record_set_ids:
    print("Attempting to discover RecordSet IDs from schema: fallback...\n")
    for rs in dataset.record_sets():
        record_set_ids.append(rs['@id'])
    print(f"Discovered record_sets: {record_set_ids}")
else:
    print(f"All record_sets: {record_set_ids}")

## 3. Data Extraction
Load the available record set(s) into pandas DataFrames. All record sets are addressed by their `@id` as discovered above. For demonstration, we'll preview the fields (columns) and the first few records for each record set.

In [ ]:
# For each record set, load its contents into a dataframe (if possible)
dataframes = {}

if not record_set_ids:
    print("No record set IDs found. Unable to extract tables.")
else:
    for rs_id in record_set_ids:
        try:
            recs = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(recs)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records from RecordSet '@id': {rs_id}")
            print("Columns:", df.columns.tolist())
            display(df.head(2))
        except Exception as e:
            print(f"Failed to load record set {rs_id}: {e}")

# For downstream analyses, pick the first loaded record set (if there are multiple)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    main_df = dataframes[main_record_set_id]
    print(f"Selected record set for analysis: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Demonstrate basic data processing steps such as filtering, normalization, and grouping. All fields are referenced by their `@id` as per Croissant best practices. Adjust the choice of numeric and grouping fields based on actual record set structure.

In [ ]:
import numpy as np

if not dataframes:
    print("No dataframes loaded. Cannot perform EDA.")
else:
    # Identify a numeric field to analyze using actual columns
    numeric_candidates = [col for col in main_df.columns if np.issubdtype(main_df[col].dtype, np.number)]
    print(f"Numeric fields in selected RecordSet: {numeric_candidates}")

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0] # Choose the first numeric field
        print(f"Using numeric field '@id': {numeric_field_id}")
        # Simple threshold: use median as threshold for demonstration
        threshold = main_df[numeric_field_id].median()
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()

        print(f"Filtered records with {numeric_field_id} > {threshold} (median):")
        print(filtered_df.head())

        # Normalize this numeric column
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # For grouping, pick a non-numeric field
        group_candidates = [col for col in main_df.columns if main_df[col].dtype == 'object']
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"Grouping by field '@id': {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'std', 'count'])
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found for demonstration.")

## 5. Visualization

Visualize the distribution of a selected numeric field (referenced by `@id`) in the record set, and display relationships between two fields if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_candidates:
    plt.figure(figsize=(7, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If at least two numeric fields, show scatterplot
    if len(numeric_candidates) >= 2:
        plt.figure(figsize=(6, 4))
        sns.scatterplot(
            data=main_df,
            x=numeric_candidates[0],
            y=numeric_candidates[1]
        )
        plt.title(f"Scatterplot: {numeric_candidates[0]} vs {numeric_candidates[1]}")
        plt.show()
else:
    print("Insufficient data for visualization.")

## 6. Conclusion
In this notebook, we leveraged the `mlcroissant` library and unique Croissant `@id` entity references to programmatically access, analyze, and visualize the FAIR^2 mass spectrometry dataset. You can now extend this workflow to run complex analyses, train models, or integrate FAIR datasets with other Croissant-compliant resources.